In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing import (
    CATEGORICAL_COLUMNS,
    NUMERICAL_COLUMNS,
    ORDINAL_COLUMNS,
    create_preprocessor,
)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "credit_default.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (30000, 24)


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## Duplicates

In [2]:
duplicate_count = df.duplicated().sum()

print("Duplicated records:", duplicate_count)
print(f"Duplicate percentage: {duplicate_count / len(df) * 100:.2f}%")

Duplicated records: 35
Duplicate percentage: 0.12%


In [3]:
duplicates = df[df.duplicated(keep=False)]

display(duplicates.head(10))

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
18,360000,2,1,1,49,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
627,230000,1,1,1,39,-1,-1,-1,-1,-1,...,660,660,660,660,660,660,660,660,660,0
839,500000,1,1,1,43,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1
1093,360000,1,2,1,41,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1
1211,160000,2,3,2,26,-1,-1,-1,-1,-1,...,390,390,390,390,390,390,390,390,390,0
1601,150000,2,1,1,31,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
1759,50000,1,2,2,26,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0
1964,150000,2,1,1,38,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1
1980,150000,2,1,1,38,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,1
2621,210000,2,1,2,39,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,0,0,0


In [4]:
feature_columns = df.drop(columns=["Y"]).columns.tolist()

# Duplicates considering only the features
duplicated_features = df[
    df.duplicated(subset=feature_columns, keep=False)
]

print("Duplicated feature records:", len(duplicated_features))

# Check whether there are groups with different targets
target_variation = (
    duplicated_features
    .groupby(feature_columns)["Y"]
    .nunique()
)

print(
    "Groups with different target values:",
    (target_variation > 1).sum()
)

Duplicated feature records: 108
Groups with different target values: 21


In [5]:
conflicting_groups = (
    duplicated_features
    .groupby(feature_columns)["Y"]
    .agg(
        customer_count="count",
        target_count="nunique",
        default_count="sum"
    )
    .query("target_count > 1")
    .reset_index()
)

display(conflicting_groups)

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X17,X18,X19,X20,X21,X22,X23,customer_count,target_count,default_count
0,20000,1,2,2,24,2,2,4,4,4,...,1650,0,0,0,0,0,0,3,2,2
1,50000,2,2,2,22,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
2,80000,1,1,2,37,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
3,80000,1,3,2,27,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
4,100000,2,3,1,53,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
5,120000,2,2,2,28,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
6,130000,2,2,2,27,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
7,150000,2,1,1,32,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
8,180000,1,1,1,45,1,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1
9,180000,2,1,2,29,-2,-2,-2,-2,-2,...,0,0,0,0,0,0,0,2,2,1


### Duplicate Analysis

- 35 exact duplicate records were identified
- 108 records share identical feature values with at least one other record
- 21 groups contain identical features but different target values
- Identical feature values do not necessarily represent the same customer
- Duplicate records will be preserved because the dataset does not provide a unique customer identifier to confirm repeated observations

**Decision:** No duplicate records will be removed at this stage.

## Train-test split

In [6]:
target_col = "Y"

X = df.drop(columns=[target_col])
y = df[target_col]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (30000, 23)
Target shape: (30000,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (24000, 23)
X_test: (6000, 23)
y_train: (24000,)
y_test: (6000,)


## Feature classification

In [8]:
categorical_columns = CATEGORICAL_COLUMNS
ordinal_columns = ORDINAL_COLUMNS
numerical_columns = NUMERICAL_COLUMNS

print("Categorical:", categorical_columns)
print("Ordinal:", ordinal_columns)
print("Numerical:", numerical_columns)

Categorical: ['X2', 'X3', 'X4']
Ordinal: ['X6', 'X7', 'X8', 'X9', 'X10', 'X11']
Numerical: ['X1', 'X5', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23']


In [9]:
classified_columns = (
    categorical_columns +
    ordinal_columns +
    numerical_columns
)

print("Total features:", len(X.columns))
print("Classified features:", len(classified_columns))
print("Missing classification:", set(X.columns) - set(classified_columns))
print("Duplicated classification:", 
      len(classified_columns) != len(set(classified_columns)))

Total features: 23
Classified features: 23
Missing classification: set()
Duplicated classification: False


In [10]:
for col in categorical_columns + ordinal_columns:
    print(f"\n{col}")
    print(X_train[col].value_counts().sort_index())


X2
X2
1     9486
2    14514
Name: count, dtype: int64

X3
X3
0       12
1     8455
2    11256
3     3903
4       97
5      235
6       42
Name: count, dtype: int64

X4
X4
0       47
1    10892
2    12806
3      255
Name: count, dtype: int64

X6
X6
-2     2198
-1     4527
 0    11781
 1     3000
 2     2138
 3      245
 4       58
 5       18
 6       10
 7        9
 8       16
Name: count, dtype: int64

X7
X7
-2     3030
-1     4845
 0    12573
 1       25
 2     3145
 3      254
 4       78
 5       21
 6       11
 7       17
 8        1
Name: count, dtype: int64

X8
X8
-2     3270
-1     4733
 0    12636
 1        3
 2     3049
 3      192
 4       60
 5       16
 6       19
 7       20
 8        2
Name: count, dtype: int64

X9
X9
-2     3477
-1     4531
 0    13180
 1        2
 2     2542
 3      143
 4       53
 5       28
 6        3
 7       40
 8        1
Name: count, dtype: int64

X10
X10
-2     3659
-1     4417
 0    13560
 2     2102
 3      142
 4       65
 5       13
 6   

## Transformer

In [11]:
preprocessor = create_preprocessor()

In [12]:
preprocessor.fit(X_train)

for col, categories in zip(
    categorical_columns,
    preprocessor.named_transformers_["categorical"].categories_,
    strict=False
):
    print(f"{col}: {len(categories)} categorias")
    print(categories)

X2: 2 categorias
[1 2]
X3: 7 categorias
[0 1 2 3 4 5 6]
X4: 4 categorias
[0 1 2 3]


In [13]:
print(
    "Features geradas:",
    preprocessor.get_feature_names_out()
)

Features geradas: ['categorical__X2_1' 'categorical__X2_2' 'categorical__X3_0'
 'categorical__X3_1' 'categorical__X3_2' 'categorical__X3_3'
 'categorical__X3_4' 'categorical__X3_5' 'categorical__X3_6'
 'categorical__X4_0' 'categorical__X4_1' 'categorical__X4_2'
 'categorical__X4_3' 'numerical__X1' 'numerical__X5' 'numerical__X12'
 'numerical__X13' 'numerical__X14' 'numerical__X15' 'numerical__X16'
 'numerical__X17' 'numerical__X18' 'numerical__X19' 'numerical__X20'
 'numerical__X21' 'numerical__X22' 'numerical__X23' 'ordinal__X6'
 'ordinal__X7' 'ordinal__X8' 'ordinal__X9' 'ordinal__X10' 'ordinal__X11']


In [14]:
assert len(classified_columns) == len(X.columns)
assert set(classified_columns) == set(X.columns)
assert len(classified_columns) == len(set(classified_columns))

print("Feature classification validated successfully!")

Feature classification validated successfully!


# Data Preprocessing — Conclusions

- Duplicate records were investigated and preserved
- Data was split into training (80%) and testing (20%) sets
- Stratified sampling was used to preserve target distribution
- Categorical variables were encoded using OneHotEncoder
- Numerical variables were standardized using StandardScaler
- Ordinal variables were preserved in their original representation
- A ColumnTransformer was created to organize preprocessing
- Preprocessing transformations were fitted using training data only

## Limitations

- Identical feature profiles may occur in both training and testing sets
- Ordinal variables may require alternative encoding strategies
- Preprocessing decisions will be evaluated through model comparison